In [1]:
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import os

In [2]:
#Add for Visuals
PALETTE   = "viridis"
ACCENT    = "#2563EB"
HIGHLIGHT = "#DC2626"
NEUTRAL   = "#64748B"
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight",
                     "savefig.facecolor": "white"})

In [3]:
#Read the data
#The data source is from the BEA for 2022-2025
raw = pd.read_csv("GDPDollarByState.csv")
raw["Description"] = raw["Description"].str.strip()
print(f"Raw shape: {raw.shape}")

Raw shape: (1406, 20)


In [4]:
raw.head(5)

,GeoFIPS,GeoName,LineCode,Description,2022:Q1,2022:Q2,2022:Q3,2022:Q4,2023:Q1,2023:Q2,2023:Q3,2023:Q4,2024:Q1,2024:Q2,2024:Q3,2024:Q4,2025:Q1,2025:Q2,2025:Q3,2025:Q4
0,0,United States *,1,All industry total,25250347.0,25861292.0,26336304.0,26770514.0,27216445.0,27530055.0,28074846.0,28424722.0,28708161.0,29147044.0,29511664.0,29825182.0,30042113.0,30485729.0,31098027.0,31422526.0
1,0,United States *,2,Private industries,22370435.0,22958308.0,23394534.0,23792288.0,24186841.0,24462441.0,24949510.0,25252382.0,25477141.0,25871320.0,26193383.0,26464842.0,26639914.0,27051641.0,27629228.0,27939572.0
2,0,United States *,3,"Agriculture, forestry, fishing and hunting",271638.0,296635.0,300780.0,307138.0,302022.0,276696.0,261042.0,242003.0,256803.0,261204.0,268639.0,292099.0,293163.0,266907.0,265971.0,262013.0
3,0,United States *,6,"Mining, quarrying, and oil and gas extraction",420912.0,513633.0,496340.0,436892.0,403631.0,386371.0,424049.0,433225.0,406190.0,418263.0,403323.0,389216.0,411077.0,376031.0,379094.0,370940.0
4,0,United States *,10,Utilities,391678.0,452398.0,464923.0,454149.0,460092.0,463011.0,464068.0,443915.0,452495.0,459787.0,450744.0,454405.0,460146.0,460838.0,478277.0,485724.0


In [5]:
Q_COLS = [c for c in raw.columns if ":" in c]
print(f"Quarters: {Q_COLS}")

Quarters: ['2022:Q1', '2022:Q2', '2022:Q3', '2022:Q4', '2023:Q1', '2023:Q2', '2023:Q3', '2023:Q4', '2024:Q1', '2024:Q2', '2024:Q3', '2024:Q4', '2025:Q1', '2025:Q2', '2025:Q3', '2025:Q4']


In [6]:
# Convert quarter values to numeric (they may contain commas or spaces)
for q in Q_COLS:
    raw[q] = pd.to_numeric(raw[q].astype(str).str.replace(",", ""), errors="coerce")

In [7]:
# Build long-format DataFrame
long = raw.melt(id_vars=["GeoFIPS", "GeoName", "LineCode", "Description"],
                value_vars=Q_COLS, var_name="Quarter", value_name="GDP_M")

In [8]:
# Create a function for reason: Parse quarter -> period index
def qstr_to_period(s):
    yr, q = s.split(":")
    month = {"Q1": 1, "Q2": 4, "Q3": 7, "Q4": 10}[q]
    return pd.Period(f"{yr}Q{q[1]}")

In [9]:
long["Period"] = long["Quarter"].apply(qstr_to_period)
long["Year"] = long["Period"].apply(lambda p: p.year)
long["Qtr"] = long["Period"].apply(lambda p: p.quarter)
long["GDP_B"] = long["GDP_M"] / 1_000          # millions -> billions
long["GDP_T"] = long["GDP_M"] / 1_000_000      # millions -> trillions
long = long.sort_values(["GeoName", "Description", "Period"])

In [10]:
#Subset: only US-level "All industry total" → national GDP time series
us_total = long[(long["GeoName"] == "United States *") &
                (long["Description"] == "All industry total")].copy()
 
# State-level "All industry total" (excluding US total and overseas)
state_total = long[(long["GeoName"] != "United States *") &
                   (~long["GeoName"].str.contains("overseas", case=False)) &
                   (long["Description"] == "All industry total")].copy()
 
# Industry breakdown at US level
us_industry = long[(long["GeoName"] == "United States *") &
                   (~long["Description"].str.contains("overseas", case=False))].copy()